# parser_dev: revised implementation plan (old vs proposed, pasteable)

This pass integrates the latest decisions:
- `_atom_info` is a dictionary with keyed fields.
- Remove `explicit_hydrogens` and `should_infer_hydrogens` from atom info.
- Collapse `isotope` + `atomic_weight` into one field.
- Collapse stereo storage into one field.
- Canonicalization should be YARP-independent logic (not RDKit-dependent canonicalization).
- Output SMILES: uppercase, explicit bond order, stereo-free.
- Map-aware XYZ/MOL export included.


## 1) Atom-info schema (dictionary, simplified keys)

### Old
```python
# list-based mixed fields
[element, formal_charge, explicit_hydrogens, isotope, atom_mapping, should_infer_hydrogens, ...]
```

### Proposed
```python
# dictionary keyed by local atom index
atom_info = {
    i: {
        "atom_index": int,              # local index in current yarpecule ordering
        "atom_map": int | None,         # AAM index (external mapping index)
        "element": str,                 # ALWAYS uppercase, e.g. 'C', 'CL'
        "formal_charge": int,
        "mass": float,                  # collapsed isotope/atomic-weight field
        "stereo": {                     # collapsed stereo field
            "atom": str | None,         # '@'/'@@'/None
            "bonds": dict               # neighbor_idx -> '/' or '\' (single ownership rule)
        },
        "aromatic_input": bool
    }
}
```

Naming note:
- `atom_map` is clearer than `map_id` and avoids overlap with `atom_index`.


## 2) `smiles2adjmat()` tokenizer update (`%10+`, clearer token intent)

### Old (`yarp/yarpecule/graph/smiles.py`)
```python
smiles2adjmat.token_pattern = r'(\[[^\]]*\]|[A-Z](?:[a-z]+)?|[a-z]|\d{1}|[=#+\-\\/.()])'
```

### Proposed
```python
# bracket atom, %nn ring index, element tokens, bond/topology tokens
smiles2adjmat.token_pattern = r'(\[[^\]]*\]|%\d{2}|[A-Z](?:[a-z]+)?|[a-z]|\d|[=#+\-:\\/.()])'
smiles2adjmat.ring_pattern = re.compile(r'^(%\d{2}|\d)$')
```


## 3) `smiles2adjmat()` atom record build (dict-based)

### Old
```python
atom_info.append([element_label, formal_charge, explicit_hydrogens, isotope, atom_mapping, should_infer_hydrogens])
```

### Proposed
```python
# per atom during parse
stereo_atom = '@@' if '@@' in token else '@' if '@' in token else None
isotope_num = int(isotope_match.group(1)) if isotope_match else None

# collapsed mass field:
# if isotope specified -> use isotope number as mass estimate
# else -> default natural atomic weight lookup
if isotope_num is not None:
    mass_value = float(isotope_num)
else:
    mass_value = float(el_mass[element_label.capitalize().lower()])

atom_info[atom_counter] = {
    "atom_index": atom_counter,
    "atom_map": atom_mapping,
    "element": element_label.upper(),
    "formal_charge": int(formal_charge),
    "mass": mass_value,
    "stereo": {"atom": stereo_atom, "bonds": {}},
    "aromatic_input": element_label.islower(),
}
```


## 4) Stereo bond storage (single ownership)

### Old
```python
# no persistent / and \ capture
```

### Proposed
```python
# store each directional bond only once to avoid duplication
owner = min(left_atom_idx, right_atom_idx)
other = max(left_atom_idx, right_atom_idx)
atom_info[owner]["stereo"]["bonds"][other] = token  # '/' or '\'
```

Rationale:
- No information loss if ownership rule is deterministic.
- Smaller payload, easier review.


## 5) Aromatic handling (non-kekulizing-first, with warning fallback)

### Old
```python
# alternating assignment over rings with remove_fused=True
```

### Proposed
```python
if any(atom_info[i]["aromatic_input"] for i in atom_info):
    try:
        adjmat = assign_aromatic_bonds(adjmat, atom_info)
    except Exception as e:
        print(f"WARNING: aromatic assignment fallback used: {e}")
        # fallback: keep graph as parsed and rely on find_lewis for acceptable resonance form
        pass
```

Comment:
- This matches your preference to avoid hard dependence on kekulization.
- `find_lewis` handles equivalent aromatic structures downstream.


## 6) Map rules in parser (preserve/validate/fill)

### Old
```python
adjmat, atom_info = reorder_by_mappings(adjmat, atom_info)
```

### Proposed
```python
provided = [atom_info[i]["atom_map"] for i in atom_info if atom_info[i]["atom_map"] is not None]
if len(provided) != len(set(provided)):
    dupes = sorted({m for m in provided if provided.count(m) > 1})
    raise ValueError(f"Duplicate atom-map indices in input SMILES: {dupes}")

used = set(provided)
next_map = 0
heavy = [i for i in atom_info if atom_info[i]["element"] != 'H']
hyd = [i for i in atom_info if atom_info[i]["element"] == 'H']
for i in heavy + hyd:
    if atom_info[i]["atom_map"] is None:
        while next_map in used:
            next_map += 1
        atom_info[i]["atom_map"] = next_map
        used.add(next_map)
        next_map += 1

adjmat, atom_info = reorder_by_mappings(adjmat, atom_info)
```


## 7) `add_hydrogens()` with dict atom_info

### Old
```python
return new_adjmat, atom_info + [['H', 0, None, None, None, True]]
```

### Proposed
```python
# append new hydrogen entries as dict items
for _ in range(int(sum(hydrogens_to_add))):
    h_idx = len(atom_info)
    atom_info[h_idx] = {
        "atom_index": h_idx,
        "atom_map": None,
        "element": "H",
        "formal_charge": 0,
        "mass": float(el_mass['h']),
        "stereo": {"atom": None, "bonds": {}},
        "aromatic_input": False,
    }

return new_adjmat, atom_info
```


## 8) `xyz_from_smiles()` returns atom_info dict upstream

### Old (`yarp/yarpecule/input_parsers.py`)
```python
return elements, geo, adj_mat, q
```

### Proposed
```python
# yarp mode
return elements, geo, adj_mat, q, atom_info

# rdkit mode: build same dict schema + assign atom_map if absent
return elements, geo, adj_mat, q, atom_info
```


## 9) `yarpecule._read_structure()` and `_order_atoms()` behavior

### Old
```python
self._elements, self._geo, self._adj_mat, self._q = xyz_from_smiles(...)
```

### Proposed
```python
self._elements, self._geo, self._adj_mat, self._q, self._atom_info = xyz_from_smiles(...)

# If no maps provided by user and canon=True:
# 1) canonical reorder first
# 2) then assign atom_map in canonical order
# If maps provided, preserve ordering/map semantics by default.
```


## 10) New module proposal for YARP-specific canonical SMILES generation

### Why new file
- Keeps `smiles.py` parser readable.
- Isolates canonical writer logic for code review and testing.

### Proposed file
`/Users/tburton2/Desktop/Repos/yarp-again/yarp/yarpecule/graph/smiles_yarp.py`

### Proposed public API
```python
def yarp_canonical_smiles(adj_mat, bond_mat, atom_info, mapped=False):
    """
    Returns canonical uppercase explicit-bond stereo-free smiles.
    mapped=True includes atom_map from atom_info.
    """
```

### Integration
- `yarpecule.get_smiles(mode='yarp')` calls this API for both canon and mapped strings.
- `mode='rdkit'` remains fallback; mapped mode still uses `atom_info['atom_map']`.


## 11) `get_smiles(mode='yarp')` output policy

### Old
```python
def get_smiles(self):
    # rdkit-only branch
```

### Proposed
```python
def get_smiles(self, mode='yarp'):
    if mode == 'rdkit':
        # canonical: keep RDKit canonical for compatibility
        self._canon_smi = Chem.MolToSmiles(
            mol1,
            canonical=True,
            kekuleSmiles=True,
            allBondsExplicit=True,
            isomericSmiles=False,   # inherently stereo-free
        )

        # mapped: use atom_info atom_map values (not RDKit index)
        for idx in range(mol2.GetNumAtoms()):
            mol2.GetAtomWithIdx(idx).SetProp('molAtomMapNumber', str(self._atom_info[idx]['atom_map']))

        self._map_smi = Chem.MolToSmiles(
            mol2,
            canonical=True,
            kekuleSmiles=True,
            allBondsExplicit=True,
            isomericSmiles=False,   # inherently stereo-free
        )

    else:
        # yarp writer must never emit stereo tokens by design
        self._canon_smi, self._map_smi = yarp_canonical_smiles_pair(self)
```

Notes:
- This removes dependence on a separate `strip_stereo` post-step in RDKit mode.
- YARP mode should also be stereo-free at emission time, not by cleanup.



## 12) Map-aware XYZ/MOL export feasibility and edits

### Is this format-compatible?
- **MOL**: yes, atom mapping is standard/compatible in atom block mapping field.
- **XYZ**: no strict atom-map standard.

### Reader compatibility choice (Avogadro/VMD friendly)
Use standard XYZ geometry lines unchanged, and put map metadata in the comment line.
This is easiest for humans and least likely to break readers.

Example XYZ header:
```text
12
maps=0,1,2,3,4,5,6,7,8,9,10,11
C ...
...
```

### Old (`yarp/util/write_files.py`)
```python
def mol_write_yp(file, elements, geo, bond_mat, adj_mat, append_opt=False):
    ...
def xyz_write(name, elements, geo, append_opt=False):
    ...
```

### Proposed
```python
def mol_write_yp(file, elements, geo, bond_mat, adj_mat, atom_info=None, append_opt=False):
    # write atom mapping field from atom_info[idx]['atom_map'] when provided
    ...


def xyz_write(name, elements, geo, atom_info=None, append_opt=False):
    # keep standard XYZ body; add "maps=..." to comment line when atom_info provided
    ...
```

### Proposed yarpecule call site
```python
if format == 'xyz':
    xyz_write(filename, self.elements, self.geo, atom_info=self._atom_info)
elif format == 'mol':
    mol_write_yp(filename, self.elements, self.geo, self.bond_mats[0], self.adj_mat, atom_info=self._atom_info)
```



## 13) Unit tests (mirror existing class-based style)

### Proposed additions (fixture-based)
```python
class TestSmi2AdjMapping:
    def test_duplicate_map_raises(self, haa_full_map_smi):
        ...

    def test_percent10_ring_token(self, benzene_smi):
        ...

class TestYarpeculeAtomInfo:
    def test_atom_info_dict(self, haa_full_map_smi):
        y = ypcule(haa_full_map_smi, mode='yarp', canon=False)
        assert isinstance(y._atom_info, dict)

    def test_element_is_uppercase(self, haa_full_map_smi):
        y = ypcule(haa_full_map_smi, mode='yarp', canon=False)
        assert all(y._atom_info[i]['element'].isupper() for i in y._atom_info)

class TestExportMapAware:
    def test_xyz_export_contains_map_metadata(self, tmp_path, haa_full_map_smi):
        ...

    def test_mol_export_contains_map_field(self, tmp_path, haa_full_map_smi):
        ...
```
```


## 14) Prompt for web version (updated)

```text
Help design a Python canonical SMILES writer with these constraints:
1) parser stores atom info as dict keyed by atom index with fields:
   atom_index, atom_map, element (uppercase), formal_charge, mass, stereo{atom,bonds}, aromatic_input
2) preserve user atom_map exactly; error on duplicate mapped input; fill missing only
3) support ring closures >9 (%10)
4) output stereo-free canonical uppercase explicit-bond SMILES and mapped variant
5) avoid toolkit-dependent canonicalization in primary path; toolkit only as optional fallback oracle
6) aromatic lowercase input should be handled deterministically with warning+fallback strategy
7) propose robust test matrix and cite primary sources
```


## 15) Open questions
1. Aromatic fallback default is warning+fallback (decided). Do we also expose `strict=True` in developer/debug mode?
2. If we later need isotope precision beyond integer override, should we add a secondary optional field (`exact_mass`) while keeping `mass` simple?
3. For disconnected canonical output, should component ordering be purely canonical-rank based or additionally sorted by formula as a tiebreak?

